# TB Detection — Audio Embedding Model Training

## Data Loading & Participant-Level Split

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    classification_report, roc_curve
)

parquetPath = 'data/recording_audio_embeddings.parquet'

df = pd.read_parquet(parquetPath)
print('Total recordings:', len(df))
print('Unique participants:', df['participant'].nunique())
print(df['tbStatus'].value_counts())

participantLabels = df.groupby('participant')['tbStatus'].first()

trainIds, testIds = train_test_split(
    participantLabels.index,
    test_size=0.2,
    stratify=participantLabels.values,
    random_state=42
)

trainDf = df[df['participant'].isin(trainIds)].reset_index(drop=True)
testDf = df[df['participant'].isin(testIds)].reset_index(drop=True)

print(f'\nTrain recordings: {len(trainDf)} | Train participants: {trainDf["participant"].nunique()}')
print(f'Test recordings:  {len(testDf)} | Test participants:  {testDf["participant"].nunique()}')

xTrain = np.stack(trainDf['audio_embedding'].values).astype(np.float32)
yTrain = trainDf['tbStatus'].astype(int).values

xTest = np.stack(testDf['audio_embedding'].values).astype(np.float32)
yTest = testDf['tbStatus'].astype(int).values

print(f'\nxTrain: {xTrain.shape} | xTest: {xTest.shape}')

neg, pos = np.bincount(yTrain)
scalePosWeight = neg / pos
print(f'\nClass balance -> neg: {neg}, pos: {pos}, scale_pos_weight: {scalePosWeight:.2f}')


def evaluate(modelName, probsTest, testDf, yTest, threshold=0.5):
    preds = (probsTest >= threshold).astype(int)

    print(f'\n===== {modelName} — RECORDING LEVEL =====')
    print('AUC-ROC:', roc_auc_score(yTest, probsTest))
    print('Accuracy:', accuracy_score(yTest, preds))
    print(confusion_matrix(yTest, preds))
    print(classification_report(yTest, preds, target_names=['Negative', 'Positive']))

    evalDf = testDf.copy()
    evalDf['prob'] = probsTest
    patientProbs = evalDf.groupby('participant').agg(
        prob=('prob', 'mean'),
        tbStatus=('tbStatus', 'first')
    )
    patientPreds = (patientProbs['prob'] >= threshold).astype(int)

    print(f'\n===== {modelName} — PATIENT LEVEL (aggregated) =====')
    print('AUC-ROC:', roc_auc_score(patientProbs['tbStatus'], patientProbs['prob']))
    print('Accuracy:', accuracy_score(patientProbs['tbStatus'], patientPreds))
    print(confusion_matrix(patientProbs['tbStatus'], patientPreds))
    print(classification_report(patientProbs['tbStatus'], patientPreds, target_names=['Negative', 'Positive']))

    return patientProbs

In [ ]:
import sys
print(sys.executable)

!{sys.executable} -m pip install xgboost

## XGBoost (Baseline)

In [ ]:
import xgboost as xgb

xgbModel = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scalePosWeight,
    eval_metric='auc',
    tree_method='hist',
    early_stopping_rounds=30,
    random_state=42,
)

xgbModel.fit(
    xTrain, yTrain,
    eval_set=[(xTest, yTest)],
    verbose=50
)

xgbProbs = xgbModel.predict_proba(xTest)[:, 1]
xgbPatientProbs = evaluate('XGBoost (Baseline)', xgbProbs, testDf, yTest)

importances = xgbModel.feature_importances_
topIdx = np.argsort(importances)[::-1][:15]
print('\nTop 15 embedding dims by importance:', topIdx)

## MLP (Baseline)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training on:', device)


class EmbeddingDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class TBClassifier(nn.Module):
    def __init__(self, inputDim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inputDim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def trainMlp(xFit, yFit, xVal, yVal, inputDim, scalePosWeight, epochs=100, patience=15, batchSize=64, lr=1e-3, verbose=False):
    fitLoader = DataLoader(EmbeddingDataset(xFit, yFit), batch_size=batchSize, shuffle=True, drop_last=True)
    valLoader = DataLoader(EmbeddingDataset(xVal, yVal), batch_size=256, shuffle=False)

    model = TBClassifier(inputDim=inputDim).to(device)
    posWeight = torch.tensor([scalePosWeight], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=posWeight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

    bestAuc = 0
    bestState = None
    patienceCounter = 0

    for epoch in range(epochs):
        model.train()
        totalLoss = 0

        for xb, yb in fitLoader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            totalLoss += loss.item() * len(xb)

        model.eval()
        allProbs, allLabels = [], []
        with torch.no_grad():
            for xb, yb in valLoader:
                xb = xb.to(device)
                probs = torch.sigmoid(model(xb)).cpu().numpy()
                allProbs.extend(probs)
                allLabels.extend(yb.numpy())

        valAuc = roc_auc_score(allLabels, allProbs)
        scheduler.step(valAuc)

        if valAuc > bestAuc:
            bestAuc = valAuc
            bestState = model.state_dict()
            patienceCounter = 0
        else:
            patienceCounter += 1

        if verbose and (epoch % 5 == 0 or patienceCounter == 0):
            print(f'Epoch {epoch:3d} | Loss: {totalLoss / len(xFit):.4f} | Val AUC: {valAuc:.4f} | Best: {bestAuc:.4f}')

        if patienceCounter >= patience:
            if verbose:
                print(f'Early stopping at epoch {epoch}')
            break

    model.load_state_dict(bestState)
    model.eval()

    with torch.no_grad():
        valTensor = torch.tensor(xVal, dtype=torch.float32).to(device)
        valProbs = torch.sigmoid(model(valTensor)).cpu().numpy()

    return model, valProbs, bestAuc


mlpModel, mlpProbs, mlpBestAuc = trainMlp(
    xTrain, yTrain, xTest, yTest,
    inputDim=xTrain.shape[1],
    scalePosWeight=scalePosWeight,
    epochs=100,
    patience=15,
    verbose=True
)

mlpPatientProbs = evaluate('MLP', mlpProbs, testDf, yTest)

## Logistic Regression (Baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
xTrainScaled = scaler.fit_transform(xTrain)
xTestScaled = scaler.transform(xTest)

logReg = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=1.0,
    random_state=42
)
logReg.fit(xTrainScaled, yTrain)

logRegProbs = logReg.predict_proba(xTestScaled)[:, 1]
logRegPatientProbs = evaluate('Logistic Regression', logRegProbs, testDf, yTest)

## Baseline Model Comparison

In [ ]:
baselineComparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost (Baseline)', 'MLP'],
    'Patient AUC': [
        roc_auc_score(logRegPatientProbs['tbStatus'], logRegPatientProbs['prob']),
        roc_auc_score(xgbPatientProbs['tbStatus'], xgbPatientProbs['prob']),
        roc_auc_score(mlpPatientProbs['tbStatus'], mlpPatientProbs['prob']),
    ]
})
print(baselineComparison.sort_values('Patient AUC', ascending=False))

## 1. Hyperparameter Tuning — XGBoost

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold

paramDistributions = {
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    'n_estimators': [200, 300, 500, 800],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0],
}

baseXgb = xgb.XGBClassifier(
    scale_pos_weight=scalePosWeight,
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
)

trainGroups = trainDf['participant'].values
groupCv = GroupKFold(n_splits=5)

xgbSearch = RandomizedSearchCV(
    estimator=baseXgb,
    param_distributions=paramDistributions,
    n_iter=40,
    scoring='roc_auc',
    cv=groupCv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

xgbSearch.fit(xTrain, yTrain, groups=trainGroups)

bestParams = xgbSearch.best_params_
print('Best CV AUC:', xgbSearch.best_score_)
print('Best params:', bestParams)

## 2. Retrain Final XGBoost with Best Params — Patient-Level Evaluation

In [ ]:
xgbFinal = xgb.XGBClassifier(
    **bestParams,
    scale_pos_weight=scalePosWeight,
    eval_metric='auc',
    tree_method='hist',
    early_stopping_rounds=30,
    random_state=42,
)

xgbFinal.fit(
    xTrain, yTrain,
    eval_set=[(xTest, yTest)],
    verbose=50
)

xgbFinalProbs = xgbFinal.predict_proba(xTest)[:, 1]
xgbFinalPatientProbs = evaluate('XGBoost (Tuned)', xgbFinalProbs, testDf, yTest)

## 3. Ensemble — XGBoost (Tuned) + MLP

In [ ]:
ensembleProbs = (xgbFinalProbs + mlpProbs) / 2
ensemblePatientProbs = evaluate('Ensemble (XGB + MLP, avg)', ensembleProbs, testDf, yTest)

## 4. Stacked Meta-Model (Replaces Manual Averaging)

In [ ]:
stackCv = GroupKFold(n_splits=5)
oofXgb = np.zeros(len(xTrain))
oofMlp = np.zeros(len(xTrain))

for foldIdx, (fitIdx, valIdx) in enumerate(stackCv.split(xTrain, yTrain, groups=trainGroups)):
    xFit, xVal = xTrain[fitIdx], xTrain[valIdx]
    yFit, yVal = yTrain[fitIdx], yTrain[valIdx]

    foldXgb = xgb.XGBClassifier(
        **bestParams,
        scale_pos_weight=scalePosWeight,
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
    )
    foldXgb.fit(xFit, yFit)
    oofXgb[valIdx] = foldXgb.predict_proba(xVal)[:, 1]

    _, foldMlpProbs, _ = trainMlp(
        xFit, yFit, xVal, yVal,
        inputDim=xTrain.shape[1],
        scalePosWeight=scalePosWeight,
        epochs=60,
        patience=10
    )
    oofMlp[valIdx] = foldMlpProbs

    print(f'Fold {foldIdx + 1}/5 done')

metaFeaturesTrain = np.column_stack([oofXgb, oofMlp])
metaFeaturesTest = np.column_stack([xgbFinalProbs, mlpProbs])

metaModel = LogisticRegression(max_iter=1000, random_state=42)
metaModel.fit(metaFeaturesTrain, yTrain)

stackedProbs = metaModel.predict_proba(metaFeaturesTest)[:, 1]
stackedPatientProbs = evaluate('Stacked Meta-Model (XGB + MLP)', stackedProbs, testDf, yTest)

## 5. Demographic Features Alongside Embeddings

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

demographicCols = ['sex', 'age', 'height', 'weight', 'country']

trainDemo = trainDf[demographicCols].copy()
testDemo = testDf[demographicCols].copy()

heightMedian = trainDemo['height'].median()
trainDemo['height'] = trainDemo['height'].fillna(heightMedian)
testDemo['height'] = testDemo['height'].fillna(heightMedian)

demoPreprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), ['age', 'height', 'weight']),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['sex', 'country']),
    ]
)

trainDemoEncoded = demoPreprocessor.fit_transform(trainDemo)
testDemoEncoded = demoPreprocessor.transform(testDemo)

xTrainWithDemo = np.hstack([xTrain, trainDemoEncoded]).astype(np.float32)
xTestWithDemo = np.hstack([xTest, testDemoEncoded]).astype(np.float32)

print('Combined feature shape:', xTrainWithDemo.shape)

xgbDemoModel = xgb.XGBClassifier(
    **bestParams,
    scale_pos_weight=scalePosWeight,
    eval_metric='auc',
    tree_method='hist',
    early_stopping_rounds=30,
    random_state=42,
)

xgbDemoModel.fit(
    xTrainWithDemo, yTrain,
    eval_set=[(xTestWithDemo, yTest)],
    verbose=50
)

xgbDemoProbs = xgbDemoModel.predict_proba(xTestWithDemo)[:, 1]
xgbDemoPatientProbs = evaluate('XGBoost + Demographics', xgbDemoProbs, testDf, yTest)

## Final Model Comparison

In [ ]:
finalComparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'XGBoost (Baseline)',
        'MLP',
        'XGBoost (Tuned)',
        'Ensemble (XGB + MLP)',
        'Stacked Meta-Model',
        'XGBoost + Demographics',
    ],
    'Patient AUC': [
        roc_auc_score(logRegPatientProbs['tbStatus'], logRegPatientProbs['prob']),
        roc_auc_score(xgbPatientProbs['tbStatus'], xgbPatientProbs['prob']),
        roc_auc_score(mlpPatientProbs['tbStatus'], mlpPatientProbs['prob']),
        roc_auc_score(xgbFinalPatientProbs['tbStatus'], xgbFinalPatientProbs['prob']),
        roc_auc_score(ensemblePatientProbs['tbStatus'], ensemblePatientProbs['prob']),
        roc_auc_score(stackedPatientProbs['tbStatus'], stackedPatientProbs['prob']),
        roc_auc_score(xgbDemoPatientProbs['tbStatus'], xgbDemoPatientProbs['prob']),
    ]
})
print(finalComparison.sort_values('Patient AUC', ascending=False))

## Save Artifacts

In [ ]:
import json
import os

import joblib
import torch

saveDir = 'output'
os.makedirs(saveDir, exist_ok=True)

joblib.dump(logReg, f'{saveDir}/tb_logreg_model.pkl')
joblib.dump(scaler, f'{saveDir}/tb_logreg_scaler.pkl')
print('Saved: tb_logreg_model.pkl, tb_logreg_scaler.pkl')

xgbFinal.save_model(f'{saveDir}/tb_xgb_model.json')
joblib.dump(xgbFinal, f'{saveDir}/tb_xgb_model.pkl')
print('Saved: tb_xgb_model.json, tb_xgb_model.pkl')

torch.save(mlpModel.state_dict(), f'{saveDir}/tb_mlp_model.pt')
mlpConfig = {'inputDim': xTrain.shape[1], 'hidden': [256, 64]}
with open(f'{saveDir}/tb_mlp_config.json', 'w') as f:
    json.dump(mlpConfig, f)
print('Saved: tb_mlp_model.pt, tb_mlp_config.json')

joblib.dump(metaModel, f'{saveDir}/tb_stack_meta_model.pkl')
print('Saved: tb_stack_meta_model.pkl')

joblib.dump(demoPreprocessor, f'{saveDir}/tb_demo_preprocessor.pkl')
xgbDemoModel.save_model(f'{saveDir}/tb_xgb_demo_model.json')
joblib.dump(xgbDemoModel, f'{saveDir}/tb_xgb_demo_model.pkl')
print('Saved: tb_demo_preprocessor.pkl, tb_xgb_demo_model.json, tb_xgb_demo_model.pkl')

print('\nAll models saved to:', saveDir)

## ONNX Export for Rust Inference (tract-compatible)

Every model below is converted to ONNX and its predictions are checked against the native model's predictions before being written to disk, so a `FAIL` here means don't ship that file.

In [ ]:
import sys

!{sys.executable} -m pip install onnx skl2onnx onnxmltools onnxruntime

In [ ]:
import onnxruntime as ort
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType as OmtFloatTensorType
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, Int64TensorType, StringTensorType
from sklearn.pipeline import Pipeline


def verifyOnnx(onnxModel, inputFeed, referenceProbs, label):
    session = ort.InferenceSession(onnxModel.SerializeToString(), providers=['CPUExecutionProvider'])
    onnxOutput = session.run(None, inputFeed)[-1]
    if isinstance(onnxOutput, list):
        print(f'{label}: FAIL — got a ZipMap/sequence output, not a tensor (tract cannot load this; set zipmap=False)')
        return onnxModel
    onnxProbs = onnxOutput[:, -1] if onnxOutput.ndim == 2 else onnxOutput.ravel()
    maxDiff = np.max(np.abs(onnxProbs - referenceProbs))
    status = 'PASS' if maxDiff < 1e-4 else 'FAIL'
    print(f'{label}: max abs diff vs native predictions = {maxDiff:.8f} [{status}]')
    return onnxModel

In [ ]:
xgbFinalOnnx = convert_xgboost(
    xgbFinal,
    initial_types=[('input', OmtFloatTensorType([None, xTrain.shape[1]]))]
)
verifyOnnx(xgbFinalOnnx, {'input': xTest}, xgbFinalProbs, 'XGBoost (Tuned)')

with open(f'{saveDir}/tb_xgb_tuned.onnx', 'wb') as f:
    f.write(xgbFinalOnnx.SerializeToString())

In [ ]:
class MlpInference(nn.Module):
    def __init__(self, baseModel):
        super().__init__()
        self.baseModel = baseModel

    def forward(self, x):
        return torch.sigmoid(self.baseModel(x))


mlpInferenceModel = MlpInference(mlpModel).to(device)
mlpInferenceModel.eval()

dummyInput = torch.randn(1, xTrain.shape[1], device=device)
torch.onnx.export(
    mlpInferenceModel,
    dummyInput,
    f'{saveDir}/tb_mlp.onnx',
    input_names=['input'],
    output_names=['probability'],
    dynamic_axes={'input': {0: 'batch'}, 'probability': {0: 'batch'}},
    opset_version=17
)

mlpOnnxSession = ort.InferenceSession(f'{saveDir}/tb_mlp.onnx', providers=['CPUExecutionProvider'])
mlpOnnxProbs = mlpOnnxSession.run(None, {'input': xTest})[0].ravel()
maxDiff = np.max(np.abs(mlpOnnxProbs - mlpProbs))
status = 'PASS' if maxDiff < 1e-4 else 'FAIL'
print(f'MLP: max abs diff vs native predictions = {maxDiff:.8f} [{status}]')

In [ ]:
logRegPipeline = Pipeline(steps=[('scaler', scaler), ('clf', logReg)])

logRegOnnx = convert_sklearn(
    logRegPipeline,
    initial_types=[('input', FloatTensorType([None, xTrain.shape[1]]))],
    options={id(logReg): {'zipmap': False}}
)
verifyOnnx(logRegOnnx, {'input': xTest}, logRegProbs, 'Logistic Regression')

with open(f'{saveDir}/tb_logreg.onnx', 'wb') as f:
    f.write(logRegOnnx.SerializeToString())

In [ ]:
metaModelOnnx = convert_sklearn(
    metaModel,
    initial_types=[('input', FloatTensorType([None, 2]))],
    options={id(metaModel): {'zipmap': False}}
)
verifyOnnx(metaModelOnnx, {'input': metaFeaturesTest.astype(np.float32)}, stackedProbs, 'Stacked Meta-Model')

with open(f'{saveDir}/tb_stack_meta_model.onnx', 'wb') as f:
    f.write(metaModelOnnx.SerializeToString())

In [ ]:
initialTypesDemo = [
    ('sex', StringTensorType([None, 1])),
    ('age', Int64TensorType([None, 1])),
    ('height', FloatTensorType([None, 1])),
    ('weight', FloatTensorType([None, 1])),
    ('country', StringTensorType([None, 1])),
]

demoPreprocessorOnnx = convert_sklearn(demoPreprocessor, initial_types=initialTypesDemo)

demoOnnxSession = ort.InferenceSession(demoPreprocessorOnnx.SerializeToString(), providers=['CPUExecutionProvider'])
demoFeedTest = {
    'sex': testDemo[['sex']].values.astype(object),
    'age': testDemo[['age']].values.astype(np.int64),
    'height': testDemo[['height']].values.astype(np.float32),
    'weight': testDemo[['weight']].values.astype(np.float32),
    'country': testDemo[['country']].values.astype(object),
}
demoOnnxOut = demoOnnxSession.run(None, demoFeedTest)[0]
maxDiff = np.max(np.abs(demoOnnxOut - testDemoEncoded))
status = 'PASS' if maxDiff < 1e-4 else 'FAIL'
print(f'Demographic preprocessor: max abs diff vs native transform = {maxDiff:.8f} [{status}]')

with open(f'{saveDir}/tb_demo_preprocessor.onnx', 'wb') as f:
    f.write(demoPreprocessorOnnx.SerializeToString())

xgbDemoOnnx = convert_xgboost(
    xgbDemoModel,
    initial_types=[('input', OmtFloatTensorType([None, xTrainWithDemo.shape[1]]))]
)
verifyOnnx(xgbDemoOnnx, {'input': xTestWithDemo}, xgbDemoProbs, 'XGBoost + Demographics')

with open(f'{saveDir}/tb_xgb_demo.onnx', 'wb') as f:
    f.write(xgbDemoOnnx.SerializeToString())

### Notes for Rust (`tract-onnx`) inference

- `tb_xgb_tuned.onnx`, `tb_logreg.onnx`, `tb_mlp.onnx`, `tb_stack_meta_model.onnx`, `tb_xgb_demo.onnx` and `tb_demo_preprocessor.onnx` are self-contained single-input ONNX graphs — load any one directly with `tract-onnx`, no XGBoost/libtorch linking needed.
- **Ensemble (XGB + MLP)** has no separate file: run `tb_xgb_tuned.onnx` and `tb_mlp.onnx` on the same embedding and average the two output probabilities.
- **XGBoost + Demographics** is two models chained: run `tb_demo_preprocessor.onnx` on the raw `sex`/`age`/`height`/`weight`/`country` fields, concatenate its output onto the 1024-dim embedding, then feed that combined vector into `tb_xgb_demo.onnx`.
- All ONNX models take/return `float32`; the demo preprocessor takes `sex`/`country` as strings and `age` as `int64`.